In [ ]:
%pip install ucimlrepo

In [81]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

In [ ]:
from ucimlrepo import fetch_ucirepo

chronic_kidney_disease = fetch_ucirepo(id=336)

# data (as pandas dataframes) 
X = chronic_kidney_disease.data.features.copy() 
y = chronic_kidney_disease.data.targets.copy() 
  
# metadata 
print(chronic_kidney_disease.metadata) 
  
# variable information 
print(chronic_kidney_disease.variables) 

In [ ]:
X.head()

In [67]:
print("There are", X.shape[0], "rows and", X.shape[1], "columns in the dataset.")

There are 400 rows and 24 columns in the dataset.


In [68]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 24 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     391 non-null    float64
 1   bp      388 non-null    float64
 2   sg      353 non-null    float64
 3   al      354 non-null    float64
 4   su      351 non-null    float64
 5   rbc     248 non-null    object 
 6   pc      335 non-null    object 
 7   pcc     396 non-null    object 
 8   ba      396 non-null    object 
 9   bgr     356 non-null    float64
 10  bu      381 non-null    float64
 11  sc      383 non-null    float64
 12  sod     313 non-null    float64
 13  pot     312 non-null    float64
 14  hemo    348 non-null    float64
 15  pcv     329 non-null    float64
 16  wbcc    294 non-null    float64
 17  rbcc    269 non-null    float64
 18  htn     398 non-null    object 
 19  dm      398 non-null    object 
 20  cad     398 non-null    object 
 21  appet   399 non-null    object 
 22  pe

In [ ]:
for col in X.select_dtypes(include='object').columns:
    print(col, X[col].unique())


In [ ]:
X['dm'].value_counts(dropna=False)

Object column check: All 10 object columns are categorical: each holds two categories (e.g. yes/no, normal/abnormal) plus missing values (nan). Only dm has a formatting error, '\tno' (“no” with a stray tab), which I’ll fix with .str.strip(). Next, I’ll convert each column to numbers (e.g. yes → 1, no → 0) and handle the missing values after the train/test split.

In [ ]:
X['dm'] = X['dm'].str.strip()
X['dm'].value_counts(dropna=False)

In [ ]:
y['class'] = y['class'].str.strip()
y.value_counts(dropna=False)

Encoding categorical columns: converting each category to 0/1 (yes → 1, no → 0; abnormal → 1, normal → 0) before the train/test split. Missing values stay as NaN.

In [ ]:
mapping = {
    "yes": 1, "no": 0,
    "abnormal": 1, "normal": 0,
    "present": 1, "notpresent": 0,
    "poor": 1, "good": 0,
}

categorical_cols = X.select_dtypes(include="object").columns
X[categorical_cols] = X[categorical_cols].apply(
    lambda col: col.str.strip().map(mapping)
)

for col in categorical_cols:
    print(f"\n{col}")
    print(X[col].value_counts(dropna=False))

In [74]:
y = y['class'].map({'ckd': 1, 'notckd': 0})
y.value_counts(dropna=False)

class
1    250
0    150
Name: count, dtype: int64

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\ny_train class proportions:")
print(y_train.value_counts(normalize=True))
print("\ny_test class proportions:")
print(y_test.value_counts(normalize=True))

X_train shape: (320, 24)
X_test shape: (80, 24)

y_train class proportions:
class
1    0.625
0    0.375
Name: proportion, dtype: float64

y_test class proportions:
class
1    0.625
0    0.375
Name: proportion, dtype: float64


Baseline: a model that always predicts the most common class (ckd), using DummyClassifier.
This baseline calls every patient “ckd”. So it catches all the ckd patients, but it gets every healthy patient wrong.

In [77]:
#Baseline model: DummyClassifier
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print("Baseline test accuracy:", baseline.score(X_test, y_test))

Baseline test accuracy: 0.625


Logistic regression: a pipeline that fills missing values with the training medians (SimpleImputer), scales the features (StandardScaler), then fits a LogisticRegression model.
Every step learns from the training set only, so no information leaks from the test set. It must beat the baseline test accuracy of 0.625.

In [79]:
#Logistic regression model: Pipeline
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model.fit(X_train, y_train)
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))


Train accuracy: 1.0
Test accuracy: 0.9875


In [82]:
#Sensitivity and specificity: what kind of mistakes the model makes on the test set
y_pred = model.predict(X_test)                                  # model's predictions for the test patients
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()       # counts of each type of right/wrong answer

print("Sensitivity:", tp / (tp + fn))                           # share of CKD patients caught
print("Specificity:", tn / (tn + fp))                           # share of healthy patients correctly cleared

Sensitivity: 0.98
Specificity: 1.0


In [ ]:
#Weights: how strongly each feature pushes towards CKD
weights = model.named_steps['logisticregression'].coef_[0]
importance = pd.Series(weights, index=X.columns).sort_values()
print(importance)